# Notebook 04 — Dashboard Interactivo: Customer Intelligence

**Dataset:** Online Retail II (UCI)  
**Fuente:** `customer_360.parquet` generado en Notebook 03

---
### Contenido
1. Setup y carga de datos
2. KPIs ejecutivos
3. Evolución temporal de ventas
4. Segmentos K-Means interactivos
5. Mapa de calor RFM
6. Predicciones BG/NBD y P_alive
7. Top clientes por CLV
8. Exportar dashboard como HTML standalone

## 0 · Setup

In [1]:
import os
from pathlib import Path

# Fijar working directory en la raíz del proyecto
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    os.chdir(notebook_dir.parent)
print('Working directory:', Path.cwd())

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.templates.default = 'plotly_white'

# Paleta de colores consistente con notebook 03
COLORS = {
    'primary':   '#2563EB',
    'secondary': '#7C3AED',
    'accent':    '#F59E0B',
    'success':   '#10B981',
    'danger':    '#EF4444',
    'neutral':   '#6B7280',
}
SEGMENT_PALETTE = [
    '#2563EB','#7C3AED','#10B981','#F59E0B',
    '#EF4444','#06B6D4','#D97706','#6B7280'
]

DATA_DIR   = Path('data/processed')
OUTPUT_DIR = Path('outputs/04_dashboard')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Setup completo.')

Working directory: c:\Users\esthe\Desktop\Repository\customer-decision-intelligence
✅ Setup completo.


## 1 · Carga de datos

In [2]:
# ── Tabla 360 (generada en notebook 03) ───────────────────────────────────────
c360 = pd.read_parquet(DATA_DIR / 'customer_360.parquet')
print(f'customer_360: {len(c360):,} clientes | columnas: {list(c360.columns)}')

# ── Transacciones originales (para evolución temporal) ────────────────────────
tx = pd.read_parquet(DATA_DIR / 'transactions_clean.parquet')
tx['InvoiceDate'] = pd.to_datetime(tx['InvoiceDate'])
tx['Revenue']     = tx['Quantity'] * tx['Price']
tx['YearMonth']   = tx['InvoiceDate'].dt.to_period('M').astype(str)
print(f'transactions:  {len(tx):,} filas | {tx["Customer ID"].nunique():,} clientes')

# Merge para tener el cluster_label en transacciones
tx = tx.merge(c360[['Customer ID','cluster_label','RFM_segment','clv_12m']], 
              on='Customer ID', how='left')

c360.head(3)

customer_360: 5,862 clientes | columnas: ['Customer ID', 'frequency', 'monetary', 'n_items', 'avg_order_value', 'recency', 'R_score', 'F_score', 'M_score', 'RFM_score', 'RFM_total', 'RFM_segment', 'cluster', 'cluster_label', 'p_alive', 'pred_purchases_4w', 'pred_purchases_12w', 'pred_purchases_26w', 'pred_purchases_52w', 'exp_avg_revenue', 'clv_12m']
transactions:  802,949 filas | 5,862 clientes


,Customer ID,frequency,monetary,n_items,avg_order_value,recency,R_score,F_score,M_score,RFM_score,...,RFM_segment,cluster,cluster_label,p_alive,pred_purchases_4w,pred_purchases_12w,pred_purchases_26w,pred_purchases_52w,exp_avg_revenue,clv_12m
0,12346.0,12,77556.46,74285,6463.038333,326,2,5,5,255,...,At Risk,0,Champions,0.619123,0.024041,0.072095,0.15610,0.311811,11083.867277,3250.396805
1,12347.0,8,5633.32,3286,704.165000,2,5,4,5,545,...,Champions,0,Champions,0.989218,0.064713,0.194011,0.41988,0.838037,720.083105,567.559783
2,12348.0,5,1658.40,2704,331.680000,75,3,4,4,344,...,Loyal,0,Champions,0.963319,0.035768,0.107235,0.23209,0.463270,363.131213,158.220240


## 2 · KPIs ejecutivos

In [8]:
kpis = [
    ('Revenue Total',    f'£{total_revenue:,.0f}',  '#2563EB', 'total histórico'),
    ('Clientes Únicos',  f'{total_customers:,}',     '#7C3AED', 'en el dataset'),
    ('Órdenes Totales',  f'{total_orders:,}',        '#10B981', 'transacciones únicas'),
    ('AOV Mediano',      f'£{avg_order_val:,.0f}',  '#F59E0B', 'por cliente'),
    ('CLV Total 12m',    f'£{total_clv:,.0f}',      '#2563EB', 'estimado BG/NBD'),
    ('Clientes Activos', f'{pct_alive:.1f}%',        '#10B981', 'P_alive >= 0.5'),
    ('Champions',        f'{champions_n:,}',          '#F59E0B', 'segmento top'),
    ('En Riesgo',        f'{at_risk_n:,}',           '#EF4444', 'requieren atención'),
]

fig = make_subplots(rows=1, cols=8, horizontal_spacing=0.01)

for i, (label, value, color, subtitle) in enumerate(kpis):
    fig.add_trace(
        go.Scatter(x=[0], y=[0], mode='markers', marker=dict(opacity=0)),
        row=1, col=i+1
    )
    xref = 'x' if i == 0 else f'x{i+1}'
    yref = 'y' if i == 0 else f'y{i+1}'

    fig.add_annotation(xref=xref, yref=yref, x=0, y=0.5,
        text=f'<b>{value}</b>', showarrow=False,
        font=dict(size=18, color=color), align='center')

    fig.add_annotation(xref=xref, yref=yref, x=0, y=0,
        text=f'<b>{label}</b>', showarrow=False,
        font=dict(size=10, color='#1E293B'), align='center')

    fig.add_annotation(xref=xref, yref=yref, x=0, y=-0.5,
        text=f'<i>{subtitle}</i>', showarrow=False,
        font=dict(size=9, color='#94A3B8'), align='center')

    axis_x = 'xaxis' if i == 0 else f'xaxis{i+1}'
    axis_y = 'yaxis' if i == 0 else f'yaxis{i+1}'
    fig.update_layout(**{
        axis_x: dict(visible=False, range=[-1, 1]),
        axis_y: dict(visible=False, range=[-1, 1]),
    })

for i in range(1, 8):
    fig.add_shape(type='line', xref='paper', yref='paper',
        x0=i/8, x1=i/8, y0=0.05, y1=0.95,
        line=dict(color='#E2E8F0', width=1))

fig.update_layout(
    title=dict(text='Customer Intelligence — KPIs Ejecutivos',
               font=dict(size=16), x=0.5, xanchor='center'),
    height=220,
    paper_bgcolor='white',
    plot_bgcolor='white',
    showlegend=False,
    margin=dict(t=55, b=15, l=5, r=5),
)

fig.show()
fig.write_html(OUTPUT_DIR / 'kpis.html')
print('✅ kpis.html guardado')

✅ kpis.html guardado


## 3 · Evolución temporal de ventas

In [10]:
# ── Revenue mensual ────────────────────────────────────────────────────────────
monthly = (
    tx.groupby('YearMonth')
    .agg(
        revenue   = ('Revenue', 'sum'),
        orders    = ('Invoice', 'nunique'),
        customers = ('Customer ID', 'nunique'),
    )
    .reset_index()
    .sort_values('YearMonth')
)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Revenue Mensual (£)', 'Órdenes Únicas por Mes',
        'Clientes Activos por Mes', 'Ticket Medio por Orden (£)'
    ],
    vertical_spacing=0.15, horizontal_spacing=0.08
)

traces = [
    (monthly['revenue'],    '#2563EB', 1, 1),
    (monthly['orders'],     '#7C3AED', 1, 2),
    (monthly['customers'],  '#10B981', 2, 1),
]

for series, color, row, col in traces:
    fig.add_trace(
        go.Scatter(
            x=monthly['YearMonth'], y=series,
            mode='lines+markers',
            line=dict(color=color, width=2.5),
            marker=dict(size=5, color=color),
            fill='tozeroy',
            fillcolor=f'rgba({int(color[1:3],16)},{int(color[3:5],16)},{int(color[5:7],16)},0.1)',
            showlegend=False
        ),
        row=row, col=col
    )

# Ticket medio (AOV) — calculado aparte
monthly_aov = (
    tx.groupby(['YearMonth','Invoice'])['Revenue'].sum()
    .reset_index()
    .groupby('YearMonth')['Revenue'].mean()
    .reset_index()
    .sort_values('YearMonth')
)
fig.add_trace(
    go.Scatter(
        x=monthly_aov['YearMonth'], y=monthly_aov['Revenue'],
        mode='lines+markers',
        line=dict(color='#F59E0B', width=2.5),
        marker=dict(size=5, color='#F59E0B'),
        fill='tozeroy',
        fillcolor='rgba(245,158,11,0.1)',
        showlegend=False
    ),
    row=2, col=2
)

fig.update_layout(
    title={'text': 'Evolución Temporal del Negocio', 'font': {'size': 20}},
    height=520,
    hovermode='x unified',
    paper_bgcolor='white'
)
fig.update_xaxes(tickangle=45)
fig.show()
fig.write_html(OUTPUT_DIR / 'temporal_evolution.html')
print('✅ temporal_evolution.html guardado')

✅ temporal_evolution.html guardado


In [11]:
# ── Revenue mensual por segmento K-Means ──────────────────────────────────────
monthly_seg = (
    tx.dropna(subset=['cluster_label'])
    .groupby(['YearMonth', 'cluster_label'])['Revenue']
    .sum()
    .reset_index()
    .sort_values('YearMonth')
)

fig = px.area(
    monthly_seg,
    x='YearMonth', y='Revenue',
    color='cluster_label',
    color_discrete_sequence=SEGMENT_PALETTE,
    labels={'Revenue': 'Revenue (£)', 'YearMonth': 'Mes', 'cluster_label': 'Segmento'},
    title='📊 Revenue Mensual por Segmento de Cliente'
)
fig.update_layout(height=420, hovermode='x unified', legend_title='Segmento K-Means')
fig.update_xaxes(tickangle=45)
fig.show()
fig.write_html(OUTPUT_DIR / 'revenue_by_segment_time.html')
print('✅ revenue_by_segment_time.html guardado')

✅ revenue_by_segment_time.html guardado


## 4 · Segmentos K-Means interactivos

In [12]:
# ── Scatter 3D: Recency, Frequency, Monetary por cluster ─────────────────────
plot_df = c360.copy()
plot_df['monetary_log'] = np.log1p(plot_df['monetary'])
plot_df['frequency_log'] = np.log1p(plot_df['frequency'])
# Limitar outliers para mejor visualización
q99 = plot_df['monetary_log'].quantile(0.99)
plot_df = plot_df[plot_df['monetary_log'] <= q99]

fig = px.scatter_3d(
    plot_df,
    x='recency',
    y='frequency_log',
    z='monetary_log',
    color='cluster_label',
    color_discrete_sequence=SEGMENT_PALETTE,
    size='avg_order_value',
    size_max=12,
    opacity=0.6,
    hover_data={
        'Customer ID': True,
        'recency': True,
        'frequency': True,
        'monetary': ':.0f',
        'RFM_segment': True,
        'frequency_log': False,
        'monetary_log': False,
        'avg_order_value': ':.0f'
    },
    labels={
        'recency': 'Recency (días)',
        'frequency_log': 'log(Frequency)',
        'monetary_log': 'log(Monetary £)',
        'cluster_label': 'Segmento'
    },
    title='🌐 Espacio RFM 3D por Segmento K-Means'
)
fig.update_layout(
    height=620,
    scene=dict(
        xaxis_title='Recency (días)',
        yaxis_title='log(Frequency)',
        zaxis_title='log(Monetary £)',
        bgcolor='#F8FAFC'
    ),
    legend_title='Segmento K-Means'
)
fig.show()
fig.write_html(OUTPUT_DIR / 'rfm_3d_scatter.html')
print('✅ rfm_3d_scatter.html guardado')

✅ rfm_3d_scatter.html guardado


In [13]:
# ── Sunburst: RFM_segment → cluster_label ─────────────────────────────────────
sunburst_df = (
    c360.groupby(['RFM_segment','cluster_label'])
    .agg(n=('Customer ID','count'), revenue=('monetary','sum'))
    .reset_index()
)

fig = px.sunburst(
    sunburst_df,
    path=['RFM_segment','cluster_label'],
    values='n',
    color='revenue',
    color_continuous_scale='Blues',
    hover_data={'revenue': ':.0f'},
    title='☀️ Distribución de Clientes: Segmento RFM → Cluster K-Means'
)
fig.update_layout(height=550, coloraxis_colorbar_title='Revenue £')
fig.show()
fig.write_html(OUTPUT_DIR / 'sunburst_segments.html')
print('✅ sunburst_segments.html guardado')

✅ sunburst_segments.html guardado


In [14]:
# ── Violin plots interactivos RFM por cluster ─────────────────────────────────
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Recency (días)', 'log(Frequency)', 'log(Monetary £)']
)

metrics_violin = [
    ('recency',      False),
    ('frequency',    True),
    ('monetary',     True),
]

clusters_sorted = sorted(c360['cluster_label'].dropna().unique())

for col_idx, (metric, log_scale) in enumerate(metrics_violin):
    for i, seg in enumerate(clusters_sorted):
        data = c360.loc[c360['cluster_label'] == seg, metric]
        if log_scale:
            data = np.log1p(data)
        fig.add_trace(
            go.Violin(
                y=data,
                name=seg,
                line_color=SEGMENT_PALETTE[i % len(SEGMENT_PALETTE)],
                fillcolor=SEGMENT_PALETTE[i % len(SEGMENT_PALETTE)],
                opacity=0.7,
                box_visible=True,
                meanline_visible=True,
                showlegend=(col_idx == 0)
            ),
            row=1, col=col_idx+1
        )

fig.update_layout(
    title={'text': '🎻 Distribución RFM por Segmento K-Means', 'font': {'size': 18}},
    height=480,
    violinmode='group',
    legend_title='Segmento'
)
fig.show()
fig.write_html(OUTPUT_DIR / 'violin_rfm.html')
print('✅ violin_rfm.html guardado')

✅ violin_rfm.html guardado


## 5 · Mapa de calor RFM

In [15]:
# ── Heatmap R vs F coloreado por Monetary mediano ─────────────────────────────
heatmap_df = (
    c360.groupby(['R_score','F_score'])['monetary']
    .median()
    .reset_index()
    .pivot(index='R_score', columns='F_score', values='monetary')
    .sort_index(ascending=False)
)

fig = go.Figure(go.Heatmap(
    z=heatmap_df.values,
    x=[f'F={c}' for c in heatmap_df.columns],
    y=[f'R={r}' for r in heatmap_df.index],
    colorscale='Blues',
    text=np.round(heatmap_df.values, 0).astype(int),
    texttemplate='£%{text}',
    textfont={'size': 11},
    hoverongaps=False,
    colorbar=dict(title='Monetary mediano (£)')
))

fig.update_layout(
    title={'text': '🗺️ Mapa de Calor RFM — Monetary Mediano por Celda R×F', 'font': {'size': 18}},
    xaxis_title='Frequency Score (1=bajo, 5=alto)',
    yaxis_title='Recency Score (5=reciente, 1=inactivo)',
    height=420,
    width=680
)
fig.show()
fig.write_html(OUTPUT_DIR / 'rfm_heatmap.html')
print('✅ rfm_heatmap.html guardado')

✅ rfm_heatmap.html guardado


In [16]:
# ── Heatmap R vs F coloreado por número de clientes ───────────────────────────
heatmap_n = (
    c360.groupby(['R_score','F_score'])['Customer ID']
    .count()
    .reset_index()
    .pivot(index='R_score', columns='F_score', values='Customer ID')
    .sort_index(ascending=False)
)

fig = go.Figure(go.Heatmap(
    z=heatmap_n.values,
    x=[f'F={c}' for c in heatmap_n.columns],
    y=[f'R={r}' for r in heatmap_n.index],
    colorscale='Purples',
    text=heatmap_n.values,
    texttemplate='%{text}',
    textfont={'size': 11},
    colorbar=dict(title='Nº clientes')
))

fig.update_layout(
    title={'text': '👥 Mapa de Calor RFM — Número de Clientes por Celda', 'font': {'size': 18}},
    xaxis_title='Frequency Score',
    yaxis_title='Recency Score',
    height=420,
    width=680
)
fig.show()
fig.write_html(OUTPUT_DIR / 'rfm_heatmap_count.html')
print('✅ rfm_heatmap_count.html guardado')

✅ rfm_heatmap_count.html guardado


## 6 · Predicciones BG/NBD y P_alive

In [18]:
# ── Distribución P_alive por segmento ─────────────────────────────────────────
if 'p_alive' in c360.columns:
    fig = px.histogram(
        c360.dropna(subset=['p_alive']),
        x='p_alive',
        color='cluster_label',
        color_discrete_sequence=SEGMENT_PALETTE,
        nbins=50,
        barmode='overlay',
        opacity=0.7,
        labels={'p_alive': 'P(alive)', 'cluster_label': 'Segmento'},
        title=' Distribución de Probabilidad de Estar Activo (P_alive) por Segmento'
    )
    fig.add_vline(x=0.5, line_dash='dash', line_color='red', 
                  annotation_text='Umbral 50%', annotation_position='top right')
    fig.update_layout(height=420, legend_title='Segmento')
    fig.show()
    fig.write_html(OUTPUT_DIR / 'palive_distribution.html')
    print('✅ palive_distribution.html guardado')
else:
    print('⚠️ Columna p_alive no encontrada en customer_360')

✅ palive_distribution.html guardado


In [19]:
# ── Scatter: P_alive vs CLV coloreado por segmento ───────────────────────────
if 'p_alive' in c360.columns and 'clv_12m' in c360.columns:
    plot_clv = (
        c360.dropna(subset=['p_alive','clv_12m'])
        .copy()
    )
    # Limitar outliers
    q98 = plot_clv['clv_12m'].quantile(0.98)
    plot_clv = plot_clv[plot_clv['clv_12m'] <= q98]

    fig = px.scatter(
        plot_clv,
        x='p_alive',
        y='clv_12m',
        color='cluster_label',
        size='frequency',
        size_max=18,
        color_discrete_sequence=SEGMENT_PALETTE,
        opacity=0.55,
        hover_data={
            'Customer ID': True,
            'p_alive': ':.3f',
            'clv_12m': ':.0f',
            'frequency': True,
            'monetary': ':.0f',
            'RFM_segment': True
        },
        labels={
            'p_alive': 'P(alive)',
            'clv_12m': 'CLV 12 meses (£)',
            'cluster_label': 'Segmento',
            'frequency': 'Nº Órdenes'
        },
        title='🎯 P_alive vs CLV 12 meses — Mapa de Valor del Cliente'
    )
    fig.add_vline(x=0.5, line_dash='dot', line_color='gray', opacity=0.5)
    fig.add_hline(
        y=plot_clv['clv_12m'].median(), 
        line_dash='dot', line_color='gray', opacity=0.5,
        annotation_text='CLV mediano'
    )
    fig.update_layout(height=540, legend_title='Segmento')
    fig.show()
    fig.write_html(OUTPUT_DIR / 'palive_vs_clv.html')
    print('✅ palive_vs_clv.html guardado')

✅ palive_vs_clv.html guardado


In [20]:
# ── Box CLV por segmento ──────────────────────────────────────────────────────
if 'clv_12m' in c360.columns:
    plot_clv2 = c360.dropna(subset=['clv_12m']).copy()
    q98 = plot_clv2['clv_12m'].quantile(0.98)
    plot_clv2 = plot_clv2[plot_clv2['clv_12m'] <= q98]

    fig = px.box(
        plot_clv2,
        x='cluster_label',
        y='clv_12m',
        color='cluster_label',
        color_discrete_sequence=SEGMENT_PALETTE,
        points='outliers',
        labels={'clv_12m': 'CLV 12 meses (£)', 'cluster_label': 'Segmento'},
        title='📦 Distribución CLV 12 meses por Segmento'
    )
    fig.update_layout(height=450, showlegend=False)
    fig.show()
    fig.write_html(OUTPUT_DIR / 'clv_by_segment_box.html')
    print('✅ clv_by_segment_box.html guardado')

✅ clv_by_segment_box.html guardado


## 7 · Top clientes por CLV

In [21]:
# ── Top 25 clientes por CLV ───────────────────────────────────────────────────
if 'clv_12m' in c360.columns:
    top25 = (
        c360.dropna(subset=['clv_12m'])
        .nlargest(25, 'clv_12m')
        [['Customer ID','cluster_label','RFM_segment','recency',
          'frequency','monetary','p_alive','clv_12m','avg_order_value']]
        .reset_index(drop=True)
    )
    top25.index += 1  # ranking desde 1

    fig = go.Figure(go.Bar(
        x=top25['clv_12m'],
        y=top25['Customer ID'].astype(str),
        orientation='h',
        marker=dict(
            color=top25['clv_12m'],
            colorscale='Blues',
            showscale=True,
            colorbar=dict(title='CLV £')
        ),
        text=[f"£{v:,.0f} | {seg}" for v, seg in zip(top25['clv_12m'], top25['cluster_label'])],
        textposition='outside',
        hovertemplate=(
            '<b>Cliente %{y}</b><br>'
            'CLV 12m: £%{x:,.0f}<br>'
            '<extra></extra>'
        )
    ))

    fig.update_layout(
        title={'text': '🏆 Top 25 Clientes por CLV 12 Meses', 'font': {'size': 18}},
        xaxis_title='CLV 12 meses (£)',
        yaxis_title='Customer ID',
        height=680,
        yaxis={'categoryorder': 'total ascending'},
        margin=dict(l=100, r=200)
    )
    fig.show()
    fig.write_html(OUTPUT_DIR / 'top25_clv.html')
    print('✅ top25_clv.html guardado')
    print('\nTop 10:')
    print(top25.head(10)[['Customer ID','cluster_label','clv_12m','p_alive','frequency','monetary']].to_string())

✅ top25_clv.html guardado

Top 10:
   Customer ID cluster_label       clv_12m   p_alive  frequency   monetary
1      16446.0     Champions  48124.283482  0.942536          2  168472.50
2      18102.0     Champions  37200.141311  0.998769        145  608821.65
3      14646.0     Champions  31915.837550  0.998957        145  526751.52
4      17450.0     Champions  20502.021465  0.995696         51  246973.09
5      14156.0     Champions  18576.571249  0.997207        151  305767.92
6      14911.0     Champions  17290.921614  0.999438        378  284577.97
7      14096.0     Champions  16088.692464  0.992350         17   53258.43
8      12415.0     Champions  12004.840993  0.990865         24  144033.37
9      13694.0     Champions  11915.138126  0.998619        143  196482.81
10     17511.0     Champions  10621.467224  0.998195         60  175603.55


In [22]:
# ── Curva de Pareto CLV ───────────────────────────────────────────────────────
if 'clv_12m' in c360.columns:
    pareto = (
        c360.dropna(subset=['clv_12m'])
        .sort_values('clv_12m', ascending=False)
        .copy()
    )
    pareto['cumulative_clv_pct'] = pareto['clv_12m'].cumsum() / pareto['clv_12m'].sum() * 100
    pareto['customer_pct']       = np.arange(1, len(pareto)+1) / len(pareto) * 100

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=pareto['customer_pct'],
        y=pareto['cumulative_clv_pct'],
        mode='lines',
        line=dict(color=COLORS['primary'], width=3),
        name='CLV acumulado',
        fill='tozeroy',
        fillcolor='rgba(37,99,235,0.1)'
    ))
    # Línea Pareto 80/20
    fig.add_hline(y=80, line_dash='dash', line_color=COLORS['danger'],
                  annotation_text='80% del CLV', annotation_position='right')
    # Punto de cruce
    idx_80 = (pareto['cumulative_clv_pct'] >= 80).idxmax()
    pct_80 = pareto.loc[idx_80, 'customer_pct']
    fig.add_vline(x=pct_80, line_dash='dot', line_color=COLORS['accent'],
                  annotation_text=f'{pct_80:.1f}% clientes', annotation_position='top')

    fig.update_layout(
        title={'text': '📐 Curva de Pareto — Concentración del CLV', 'font': {'size': 18}},
        xaxis_title='% de Clientes (ordenados por CLV desc)',
        yaxis_title='% CLV Acumulado',
        height=440
    )
    fig.show()
    fig.write_html(OUTPUT_DIR / 'pareto_clv.html')
    print(f'✅ pareto_clv.html guardado')
    print(f'   → El {pct_80:.1f}% de clientes genera el 80% del CLV total')

✅ pareto_clv.html guardado
   → El 28.7% de clientes genera el 80% del CLV total


## 8 · Dashboard HTML standalone completo

In [23]:
# ── Generar un único HTML con todos los gráficos ──────────────────────────────
from plotly.io import to_html

def make_all_figures():
    """Regenera todos los figures para el HTML final."""
    figs = {}

    # 1. Revenue mensual
    figs['temporal'] = px.area(
        monthly_seg, x='YearMonth', y='Revenue', color='cluster_label',
        color_discrete_sequence=SEGMENT_PALETTE,
        title='📈 Revenue Mensual por Segmento',
        labels={'Revenue':'Revenue (£)','YearMonth':'Mes','cluster_label':'Segmento'}
    )
    figs['temporal'].update_layout(height=380, hovermode='x unified')

    # 2. Sunburst
    figs['sunburst'] = px.sunburst(
        sunburst_df, path=['RFM_segment','cluster_label'], values='n',
        color='revenue', color_continuous_scale='Blues',
        title='☀️ Segmentos RFM → Cluster'
    )
    figs['sunburst'].update_layout(height=480)

    # 3. Scatter 3D RFM
    figs['scatter3d'] = px.scatter_3d(
        plot_df, x='recency', y='frequency_log', z='monetary_log',
        color='cluster_label', color_discrete_sequence=SEGMENT_PALETTE,
        opacity=0.55, size='avg_order_value', size_max=10,
        title='🌐 Espacio RFM 3D'
    )
    figs['scatter3d'].update_layout(height=550)

    # 4. Heatmap RFM
    figs['heatmap'] = go.Figure(go.Heatmap(
        z=heatmap_df.values,
        x=[f'F={c}' for c in heatmap_df.columns],
        y=[f'R={r}' for r in heatmap_df.index],
        colorscale='Blues',
        text=np.round(heatmap_df.values,0).astype(int),
        texttemplate='£%{text}',
    ))
    figs['heatmap'].update_layout(
        title='🗺️ Mapa de Calor RFM — Monetary Mediano',
        height=380
    )

    # 5. CLV scatter
    if 'p_alive' in c360.columns and 'clv_12m' in c360.columns:
        figs['palive_clv'] = px.scatter(
            plot_clv, x='p_alive', y='clv_12m',
            color='cluster_label', size='frequency',
            color_discrete_sequence=SEGMENT_PALETTE,
            opacity=0.55, size_max=15,
            title='🎯 P_alive vs CLV 12 meses',
            labels={'p_alive':'P(alive)','clv_12m':'CLV 12m (£)','cluster_label':'Segmento'}
        )
        figs['palive_clv'].update_layout(height=430)

    # 6. Top 25 CLV bar
    if 'clv_12m' in c360.columns:
        figs['top25'] = go.Figure(go.Bar(
            x=top25['clv_12m'],
            y=top25['Customer ID'].astype(str),
            orientation='h',
            marker=dict(color=top25['clv_12m'], colorscale='Blues', showscale=True),
            text=[f'£{v:,.0f}' for v in top25['clv_12m']],
            textposition='outside'
        ))
        figs['top25'].update_layout(
            title='🏆 Top 25 Clientes por CLV',
            height=620,
            yaxis={'categoryorder':'total ascending'},
            margin=dict(l=100, r=160)
        )

    # 7. Pareto
    figs['pareto'] = go.Figure(go.Scatter(
        x=pareto['customer_pct'], y=pareto['cumulative_clv_pct'],
        mode='lines', line=dict(color=COLORS['primary'], width=3),
        fill='tozeroy', fillcolor='rgba(37,99,235,0.1)'
    ))
    figs['pareto'].add_hline(y=80, line_dash='dash', line_color=COLORS['danger'])
    figs['pareto'].update_layout(
        title='📐 Curva de Pareto CLV',
        xaxis_title='% Clientes', yaxis_title='% CLV acumulado',
        height=400
    )

    return figs

all_figs = make_all_figures()
print(f'✅ {len(all_figs)} gráficos listos para el HTML')

✅ 7 gráficos listos para el HTML


In [26]:
# ── Construir el HTML completo ─────────────────────────────────────────────────
DASHBOARD_TITLE = 'Customer Intelligence Dashboard — Online Retail II'

section_order = [
    ('temporal',   '📈 Evolución Temporal'),
    ('sunburst',   '☀️ Distribución de Segmentos'),
    ('scatter3d',  '🌐 Espacio RFM 3D'),
    ('heatmap',    '🗺️ Mapa de Calor RFM'),
    ('palive_clv', '🎯 P_alive vs CLV'),
    ('top25',      '🏆 Top 25 Clientes CLV'),
    ('pareto',     '📐 Pareto CLV'),
]

# KPI cards HTML
kpi_data = [
    ('Revenue Total',    f'£{total_revenue:,.0f}',  '#2563EB'),
    ('Clientes Únicos',  f'{total_customers:,}',     '#7C3AED'),
    ('Órdenes Totales',  f'{total_orders:,}',        '#10B981'),
    ('AOV Mediano',      f'£{avg_order_val:,.0f}',  '#F59E0B'),
    ('CLV Total 12m',    f'£{total_clv:,.0f}',      '#2563EB'),
    ('Clientes Activos', f'{pct_alive:.1f}%',        '#10B981'),
    ('Champions',        f'{champions_n:,}',          '#F59E0B'),
    ('En Riesgo',        f'{at_risk_n:,}',           '#EF4444'),
]

kpi_html = ''.join([
    f'''
    <div class="kpi-card">
        <div class="kpi-value" style="color:{color}">{value}</div>
        <div class="kpi-label">{label}</div>
    </div>
    '''
    for label, value, color in kpi_data
])


charts_html = ''
for key, section_title in section_order:
    if key in all_figs:
        chart_div = to_html(all_figs[key], full_html=False, include_plotlyjs=False)
        charts_html += f"""
        <div class="section">
            <h2>{section_title}</h2>
            {chart_div}
        </div>
        """

html_content = f"""
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{DASHBOARD_TITLE}</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
    <style>
        * {{ box-sizing: border-box; margin: 0; padding: 0; }}
        body {{
            font-family: 'Inter', sans-serif;
            background: #F1F5F9;
            color: #1E293B;
        }}
        header {{
            background: linear-gradient(135deg, #1E40AF 0%, #7C3AED 100%);
            color: white;
            padding: 32px 40px;
        }}
        header h1 {{ font-size: 1.9rem; font-weight: 700; margin-bottom: 6px; }}
        header p  {{ opacity: 0.85; font-size: 0.95rem; }}
        .container {{ max-width: 1400px; margin: 0 auto; padding: 32px 24px; }}
        .kpi-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(150px, 1fr));
            gap: 16px;
            margin-bottom: 32px;
        }}
        .kpi-card {{
            background: white;
            border-radius: 12px;
            padding: 20px 16px;
            text-align: center;
            box-shadow: 0 1px 3px rgba(0,0,0,.08);
            transition: transform .15s;
        }}
        .kpi-card:hover {{ transform: translateY(-2px); box-shadow: 0 4px 12px rgba(0,0,0,.12); }}
        .kpi-icon  {{ font-size: 1.8rem; margin-bottom: 8px; }}
        .kpi-value {{ font-size: 1.5rem; font-weight: 700; margin-bottom: 4px; }}
        .kpi-label {{ font-size: 0.78rem; color: #64748B; font-weight: 500; text-transform: uppercase; letter-spacing: .05em; }}
        .section {{
            background: white;
            border-radius: 14px;
            padding: 24px;
            margin-bottom: 24px;
            box-shadow: 0 1px 3px rgba(0,0,0,.07);
        }}
        .section h2 {{
            font-size: 1.1rem;
            font-weight: 600;
            color: #1E293B;
            margin-bottom: 16px;
            padding-bottom: 10px;
            border-bottom: 2px solid #E2E8F0;
        }}
        footer {{
            text-align: center;
            padding: 24px;
            color: #94A3B8;
            font-size: 0.82rem;
        }}
    </style>
</head>
<body>
    <header>
        <h1>📊 {DASHBOARD_TITLE}</h1>
        <p>Segmentación RFM · K-Means · BG/NBD · CLV — Dataset Online Retail II (UCI) | Dic 2009 – Dic 2011</p>
    </header>
    <div class="container">
        <div class="kpi-grid">
            {kpi_html}
        </div>
        {charts_html}
    </div>
    <footer>
        Customer Intelligence Portfolio · Online Retail II (UCI) · Generado con Python, Plotly & lifetimes
    </footer>
</body>
</html>
"""

dashboard_path = OUTPUT_DIR / 'dashboard.html'
with open(dashboard_path, 'w', encoding='utf-8') as f:
    f.write(html_content)

size_kb = dashboard_path.stat().st_size / 1024
print(f'✅ Dashboard completo guardado → {dashboard_path}')
print(f'   Tamaño: {size_kb:.0f} KB')
print(f'   Ábrelo directamente en el navegador: file://{dashboard_path.resolve()}')

✅ Dashboard completo guardado → outputs\04_dashboard\dashboard.html
   Tamaño: 487 KB
   Ábrelo directamente en el navegador: file://C:\Users\esthe\Desktop\Repository\customer-decision-intelligence\outputs\04_dashboard\dashboard.html


In [27]:
# ── Resumen de archivos generados ─────────────────────────────────────────────
print('Archivos generados en outputs/04_dashboard/:')
for f in sorted(OUTPUT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'   {f.name:<45} {size_kb:>7.1f} KB')

print('\n✅ Notebook 04 completado.')
print('   → Abre outputs/04_dashboard/dashboard.html en el navegador')
print('   → Siguiente paso: Notebook 05 o app Streamlit (ver app/streamlit_app.py)')

Archivos generados en outputs/04_dashboard/:
   clv_by_segment_box.html                        4833.4 KB
   dashboard.html                                  486.7 KB
   kpis.html                                      4748.8 KB
   palive_distribution.html                       4818.4 KB
   palive_vs_clv.html                             4988.0 KB
   pareto_clv.html                                4835.5 KB
   revenue_by_segment_time.html                   4743.9 KB
   rfm_3d_scatter.html                            5226.8 KB
   rfm_heatmap.html                               4742.9 KB
   rfm_heatmap_count.html                         4742.6 KB
   sunburst_segments.html                         4744.3 KB
   temporal_evolution.html                        4745.6 KB
   top25_clv.html                                 4744.0 KB
   violin_rfm.html                                4903.4 KB

✅ Notebook 04 completado.
   → Abre outputs/04_dashboard/dashboard.html en el navegador
   → Siguiente paso: Noteb